# EnterpriseSim: GRPO Training for Customer Support Agents

Train an LLM to handle customer support using [GRPO](https://arxiv.org/abs/2402.03300) (Group Relative Policy Optimization).

The environment runs remotely on [HuggingFace Spaces](https://huggingface.co/spaces/jjmachan/enterprise-sim-support) — we connect to it, collect trajectories, and train locally with TRL.

## 1. Install Dependencies

In [ ]:
!pip install -q trl peft transformers datasets openai torch accelerate
!pip install -q "openenv-core[core]>=0.2.1"

## 2. Connect to the Environment

The EnterpriseSim environment runs on HuggingFace Spaces. We connect via WebSocket using the OpenEnv client. The agent has 4 tools: `lookup_customer`, `check_order`, `send_reply`, `update_ticket`.

In [ ]:
from typing import Any, Dict
from openenv.core.env_server.mcp_types import CallToolAction
from openenv.core.mcp_client import MCPToolClient

ENV_URL = "https://jjmachan-enterprise-sim-support.hf.space"


class SupportStepResult:
    """Rich step result exposing all observation fields."""
    def __init__(self, raw: Dict[str, Any]):
        obs = raw.get("observation", {})
        self.customer_message: str = obs.get("customer_message", "")
        self.tool_result: str = obs.get("tool_result", "")
        self.tool_name: str = obs.get("tool_name", "")
        self.ticket_context: str = obs.get("ticket_context", "")
        self.ticket_id: int = obs.get("ticket_id", 0)
        self.customer_id: str = obs.get("customer_id", "")
        self.satisfaction: float = obs.get("satisfaction", 0.0)
        self.satisfaction_delta: float = obs.get("satisfaction_delta", 0.0)
        self.resolved: bool = obs.get("resolved", False)
        self.step_count: int = obs.get("step_count", 0)
        self.episode_id: str = obs.get("episode_id", "")
        self.reward: float = raw.get("reward", 0.0)
        self.done: bool = raw.get("done", False)


class CustomerSupportEnv(MCPToolClient):
    """Client for the EnterpriseSim customer support environment."""
    def reset(self, **kwargs) -> SupportStepResult:
        response = self._send_and_receive({"type": "reset", "data": kwargs})
        return SupportStepResult(response.get("data", {}))

    def call_tool(self, name: str, **kwargs) -> SupportStepResult:
        action = CallToolAction(tool_name=name, arguments=kwargs)
        msg = {"type": "step", "data": self._step_payload(action)}
        response = self._send_and_receive(msg)
        return SupportStepResult(response.get("data", {}))

In [ ]:
# Quick test — reset and take a couple of actions
env = CustomerSupportEnv(base_url=ENV_URL)
with env:
    obs = env.reset()
    print("=== New Episode ===")
    print(f"Customer: {obs.customer_message}")
    print(f"Ticket: {obs.ticket_id} | Customer: {obs.customer_id} | Satisfaction: {obs.satisfaction}")
    print()

    obs = env.call_tool("lookup_customer", customer_id=obs.customer_id)
    print("=== lookup_customer ===")
    print(obs.tool_result[:500])
    print()

    obs = env.call_tool("send_reply", ticket_id=obs.ticket_id,
                        message="Hi! I've pulled up your account. Let me check on that order for you.")
    print("=== send_reply ===")
    print(f"Customer: {obs.customer_message}")
    print(f"Satisfaction: {obs.satisfaction} (delta: {obs.satisfaction_delta})")
    print(f"Done: {obs.done} | Resolved: {obs.resolved}")

## 3. Collect Trajectories

Run full episodes against the environment. Each step captures the conversation so far (prompt) and what the agent did (completion). This becomes our GRPO training dataset.

In [ ]:
import re, json, time
from datasets import Dataset

# --- XML tool call format used by the agent ---
TOOL_CALL_RE = re.compile(
    r"<tool_call>\s*<function=(\w+)>(.*?)</function>\s*</tool_call>", re.DOTALL
)
PARAM_RE = re.compile(r"<parameter=(\w+)>(.*?)</parameter>", re.DOTALL)
VALID_TOOLS = {"lookup_customer", "check_order", "send_reply", "update_ticket"}


def parse_tool_call(text):
    match = TOOL_CALL_RE.search(text)
    if not match:
        return None, None
    name = match.group(1).strip()
    args = {}
    for pm in PARAM_RE.finditer(match.group(2)):
        k, v = pm.group(1).strip(), pm.group(2).strip()
        if k == "ticket_id":
            try: v = int(v)
            except ValueError: pass
        args[k] = v
    return name, args


SYSTEM_PROMPT = """You are a Customer Support Representative at Office Furniture Co.

## Available Tools
- lookup_customer(customer_id, customer_name) — look up customer profile
- check_order(order_id) — get order details, shipping, tracking
- send_reply(ticket_id, message) — reply to customer (triggers their response)
- update_ticket(ticket_id, status, notes) — update ticket status

## Policies
- 30-day returns, 15% restocking fee for opened items
- Damaged items: free replacement within 48h
- 5-year warranty on furniture, 1-year accessories
- VIP: priority support, free express, 60-day returns
- $200 refund limit; escalate above that

## Response Format
Use EXACTLY this XML format:
<tool_call>
<function=TOOL_NAME>
<parameter=PARAM_NAME>value</parameter>
</function>
</tool_call>

Strategy: look up customer → check order → send helpful reply → resolve ticket."""


def format_initial_obs(obs):
    return f"New ticket: {obs.ticket_context}\n\nCustomer: {obs.customer_message}\n\nWhat tool will you use?"


def format_step_obs(obs):
    parts = []
    if obs.tool_name:
        parts.append(f'Tool "{obs.tool_name}" result:\n{obs.tool_result or "(no result)"}')
    if obs.customer_message:
        parts.append(f"Customer responded:\n{obs.customer_message}")
    parts.append(f"Satisfaction: {obs.satisfaction:.0%} | Steps: {obs.step_count}/10\nWhat next?")
    return "\n\n".join(parts)


def run_episode(env, generate_fn, seed=None):
    """Run one episode, return list of step dicts."""
    obs = env.reset(seed=seed) if seed else env.reset()
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": format_initial_obs(obs)},
    ]
    steps, ticket_id = [], obs.ticket_id

    while not obs.done:
        prompt_snapshot = [dict(m) for m in messages]
        response = generate_fn(messages)

        tool_name, tool_args = parse_tool_call(response)
        if tool_name is None:
            tool_name, tool_args = "send_reply", {"ticket_id": ticket_id, "message": response[:500]}
        if tool_name in ("send_reply", "update_ticket") and "ticket_id" not in tool_args:
            tool_args["ticket_id"] = ticket_id

        obs = env.call_tool(tool_name, **tool_args)
        steps.append({
            "prompt": prompt_snapshot, "completion": response,
            "tool_name": tool_name, "satisfaction": obs.satisfaction,
            "resolved": obs.resolved, "done": obs.done, "reward": obs.reward,
        })
        messages.append({"role": "assistant", "content": response})
        if not obs.done:
            messages.append({"role": "user", "content": format_step_obs(obs)})

    # Backfill final reward
    for s in steps:
        s["episode_reward"] = obs.reward
        s["episode_resolved"] = obs.resolved
    return steps

In [ ]:
# Collect trajectories using the Qwen model we'll also train.
# On first run this uses the base (untrained) model as behavior policy.
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-8B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.bfloat16, device_map="auto"
)

def generate_fn(messages):
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, do_sample=True, top_p=0.9)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(f"Loaded {MODEL_NAME} for trajectory collection")

In [ ]:
NUM_EPISODES = 8

all_steps = []
env = CustomerSupportEnv(base_url=ENV_URL)

with env:
    for i in range(NUM_EPISODES):
        t0 = time.time()
        steps = run_episode(env, generate_fn, seed=42 + i)
        elapsed = time.time() - t0
        all_steps.extend(steps)

        final = steps[-1] if steps else {}
        status = "RESOLVED" if final.get("episode_resolved") else "not resolved"
        print(f"[{i+1}/{NUM_EPISODES}] {len(steps)} steps, "
              f"reward={final.get('episode_reward', 0):.3f}, {status}, {elapsed:.1f}s")

# Format as GRPO dataset
records = []
for step in all_steps:
    answer = json.dumps({
        "episode_reward": step["episode_reward"],
        "resolved": step["episode_resolved"],
        "valid_tools": list(VALID_TOOLS),
    })
    records.append({"prompt": step["prompt"], "answer": answer})

dataset = Dataset.from_list(records)
print(f"\nDataset: {len(dataset)} training examples from {NUM_EPISODES} episodes")

Let's look at a sample trajectory to see what the agent is doing:

In [ ]:
# Show a sample trajectory
episode_steps = [s for s in all_steps if s["episode_reward"] == all_steps[0]["episode_reward"]]
# just the first episode's steps

print(f"Episode: {len(episode_steps)} steps | Reward: {episode_steps[0]['episode_reward']:.3f} | "
      f"Resolved: {episode_steps[0]['episode_resolved']}\n")

for i, step in enumerate(episode_steps):
    tool_name, _ = parse_tool_call(step["completion"])
    print(f"Step {i+1}: {tool_name or '(raw text)'} | satisfaction={step['satisfaction']:.2f}")
    # Show the agent's reasoning (text before tool call)
    tc_pos = step["completion"].find("<tool_call>")
    if tc_pos > 0:
        print(f"  Reasoning: {step['completion'][:tc_pos].strip()[:150]}")
    print()

## 4. GRPO Training

Train with TRL's `GRPOTrainer`. For each prompt the trainer generates multiple completions, scores them with reward functions, and optimizes toward the better ones.

**Reward functions** capture what good support looks like:
- `format_reward` — valid XML tool call syntax
- `tool_validity_reward` — uses a real tool name
- `reasoning_reward` — thinks before acting
- `no_reasoning_leak_reward` — doesn't leak internal reasoning to customer
- `action_quality_reward` — references relevant ground truth values

In [ ]:
# --- Reward Functions ---

def _get_text(c):
    return c[0]["content"] if isinstance(c, list) else str(c)

def _parse_answer(a):
    if isinstance(a, str):
        try: return json.loads(a)
        except: return {}
    return a if isinstance(a, dict) else {}


def format_reward(completions, **kw):
    """Valid XML tool call? +1.0 valid, +0.25 partial, -1.0 none"""
    rewards = []
    for c in completions:
        t = _get_text(c)
        if not t.strip(): rewards.append(-1.0); continue
        m = TOOL_CALL_RE.search(t)
        if m: rewards.append(1.0 if PARAM_RE.findall(m.group(2)) else 0.5)
        elif "<tool_call>" in t: rewards.append(0.25)
        else: rewards.append(-1.0)
    return rewards


def tool_validity_reward(completions, answer=None, **kw):
    """Valid tool name? +1.0 yes, -0.5 wrong name, 0.0 no call"""
    rewards, answers = [], answer or [None]*len(completions)
    for c, a in zip(completions, answers):
        valid = set((_parse_answer(a) if a else {}).get("valid_tools", VALID_TOOLS))
        m = TOOL_CALL_RE.search(_get_text(c))
        if m: rewards.append(1.0 if m.group(1).strip() in valid else -0.5)
        else: rewards.append(0.0)
    return rewards


def reasoning_reward(completions, **kw):
    """Reasoning before tool call? +1.0 yes (>30 chars), +0.3 call only, -0.5 none"""
    rewards = []
    for c in completions:
        pos = _get_text(c).find("<tool_call>")
        if pos > 30: rewards.append(1.0)
        elif pos >= 0: rewards.append(0.3)
        else: rewards.append(-0.5)
    return rewards


def no_reasoning_leak_reward(completions, **kw):
    """Don't leak reasoning into send_reply. +1.0 clean, -1.0 leaked, 0.0 n/a"""
    bad_starts = ["The customer", "I need to", "Let me", "From the", "Based on",
                  "Looking at", "According to", "I should", "First,", "Now I", "I'll", "I will"]
    rewards = []
    for c in completions:
        m = TOOL_CALL_RE.search(_get_text(c))
        if m and m.group(1).strip() == "send_reply":
            msg = dict(PARAM_RE.findall(m.group(2))).get("message", "").strip()
            rewards.append(-1.0 if any(msg.startswith(p) for p in bad_starts) else 1.0)
        else: rewards.append(0.0)
    return rewards


def action_quality_reward(completions, answer=None, **kw):
    """References ground truth values? +0.3 each, capped at 1.0"""
    rewards, answers = [], answer or [None]*len(completions)
    for c, a in zip(completions, answers):
        t, gts = _get_text(c), (_parse_answer(a) if a else {}).get("ground_truth_values", [])
        score = sum(0.3 for gt in gts if gt and str(gt).lower() in t.lower())
        rewards.append(min(1.0, score))
    return rewards


reward_funcs = [format_reward, tool_validity_reward, reasoning_reward,
                no_reasoning_leak_reward, action_quality_reward]
reward_weights = [1.0, 0.8, 0.5, 0.7, 0.5]

In [ ]:
from peft import LoraConfig
from trl import GRPOTrainer, GRPOConfig

# Free the inference model from GPU — the trainer will load its own copy
del model
torch.cuda.empty_cache()

peft_config = LoraConfig(
    r=32, lora_alpha=16, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

training_args = GRPOConfig(
    output_dir="./outputs/grpo-support-agent",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    num_generations=4,
    max_completion_length=512,
    temperature=0.7,
    beta=0.0,
    loss_type="grpo",
    scale_rewards="group",
    bf16=True,
    logging_steps=5,
    save_steps=100,
    log_completions=True,
    max_grad_norm=1.0,
    warmup_ratio=0.1,
)

trainer = GRPOTrainer(
    model=MODEL_NAME,
    processing_class=tokenizer,
    args=training_args,
    reward_funcs=reward_funcs,
    reward_weights=reward_weights,
    train_dataset=dataset,
    peft_config=peft_config,
)

trainer.train()
trainer.save_model("./outputs/grpo-support-agent/final_adapter")
print("Training complete! Adapter saved.")